In [16]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
# from skimage.metrics import structural_similarity as ssim
# from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy.linalg import sqrtm
from PIL import Image
import torch
from torchvision import transforms
from torchvision.models import inception_v3

import numpy as np
import torch
from torchvision import models, transforms
from PIL import Image
from scipy.linalg import sqrtm
from tqdm import tqdm


In [ ]:
ls "/mnt/Internal/MedImage/"

In [17]:
train_df_full = pd.read_csv("/mnt/Internal/MedImage/merged_dataset-Copy1.csv")


train_df_full = pd.get_dummies(train_df_full, columns=["GENDER", "PRIMARY_RACE", "ETHNICITY"])



# Convert categorical columns to integers in train_df_full

# Race
train_df_full['PRIMARY_RACE_American Indian or Alaska Native'] = train_df_full['PRIMARY_RACE_American Indian or Alaska Native'].astype(int)
train_df_full['PRIMARY_RACE_Asian'] = train_df_full['PRIMARY_RACE_Asian'].astype(int)
train_df_full['PRIMARY_RACE_Asian - Historical Conv'] = train_df_full['PRIMARY_RACE_Asian - Historical Conv'].astype(int)
train_df_full['PRIMARY_RACE_Asian, Hispanic'] = train_df_full['PRIMARY_RACE_Asian, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Asian, non-Hispanic'] = train_df_full['PRIMARY_RACE_Asian, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black or African American'] = train_df_full['PRIMARY_RACE_Black or African American'].astype(int)
train_df_full['PRIMARY_RACE_Black, Hispanic'] = train_df_full['PRIMARY_RACE_Black, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black, non-Hispanic'] = train_df_full['PRIMARY_RACE_Black, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, Hispanic'] = train_df_full['PRIMARY_RACE_Native American, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, non-Hispanic'] = train_df_full['PRIMARY_RACE_Native American, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'] = train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'].astype(int)
train_df_full['PRIMARY_RACE_Other'] = train_df_full['PRIMARY_RACE_Other'].astype(int)
train_df_full['PRIMARY_RACE_Other, Hispanic'] = train_df_full['PRIMARY_RACE_Other, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Other, non-Hispanic'] = train_df_full['PRIMARY_RACE_Other, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Patient Refused'] = train_df_full['PRIMARY_RACE_Patient Refused'].astype(int)
train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'] = train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'].astype(int)
train_df_full['PRIMARY_RACE_Unknown'] = train_df_full['PRIMARY_RACE_Unknown'].astype(int)
train_df_full['PRIMARY_RACE_White'] = train_df_full['PRIMARY_RACE_White'].astype(int)
train_df_full['PRIMARY_RACE_White or Caucasian'] = train_df_full['PRIMARY_RACE_White or Caucasian'].astype(int)
train_df_full['PRIMARY_RACE_White, Hispanic'] = train_df_full['PRIMARY_RACE_White, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_White, non-Hispanic'] = train_df_full['PRIMARY_RACE_White, non-Hispanic'].astype(int)

# Ethnicity
#train_df_full['ETHNICITY_0'] = train_df_full['ETHNICITY_0'].astype(int)
train_df_full['ETHNICITY_Hispanic'] = train_df_full['ETHNICITY_Hispanic'].astype(int)
train_df_full['ETHNICITY_Hispanic/Latino'] = train_df_full['ETHNICITY_Hispanic/Latino'].astype(int)
train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'] = train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'].astype(int)
train_df_full['ETHNICITY_Not Hispanic'] = train_df_full['ETHNICITY_Not Hispanic'].astype(int)
train_df_full['ETHNICITY_Patient Refused'] = train_df_full['ETHNICITY_Patient Refused'].astype(int)

# Gender
train_df_full['GENDER_Male'] = train_df_full['GENDER_Male'].astype(int)
train_df_full['GENDER_Female'] = train_df_full['GENDER_Female'].astype(int)

# Display the updated DataFrame
#train_df_full.head()
train_df_full.replace(-1, 1, inplace=True)

train_df_full.replace(np.nan, 0, inplace=True)



train_df_full= train_df_full[train_df_full['Frontal/Lateral'].str.contains('frontal', case=False, na=False)]


csv_file = train_df_full

# Select the first 5000 images
csv_file = csv_file.iloc[0:21970]

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from scipy.linalg import sqrtm

# Paths to datasets
original_dataset = "/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train"
generated_dataset = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_Model_Imagea_generation_with_prompt/CheXpert-v1.0/train"

def extract_features(image_paths, model, device):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),  # Inception expects 299x299 input
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    features = []

    for image_path in tqdm(image_paths, desc="Extracting Features"):
        try:
            # Open image and convert to grayscale
            image = Image.open(image_path).convert("L")
            
            # Convert grayscale to RGB by duplicating the channel
            image = Image.merge("RGB", (image, image, image))
            
            # Apply transforms and extract features
            image = transform(image).unsqueeze(0).to(device)
            with torch.no_grad():
                feature = model(image).cpu().numpy().squeeze()
            features.append(feature)
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
    
    return np.array(features)

def calculate_fid(features1, features2):
    if features1.shape[0] == 0 or features2.shape[0] == 0:
        print("Error: One of the feature sets is empty!")
        return None

    mu1, sigma1 = np.mean(features1, axis=0), np.cov(features1, rowvar=False)
    mu2, sigma2 = np.mean(features2, axis=0), np.cov(features2, rowvar=False)

    diff = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))

    # Numerical stability
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

def get_image_paths(directory, limit=None):
    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")  # Add extensions as needed
    image_paths = []

    for root, _, files in os.walk(directory):  # Recursively traverse directories
        for file in files:
            if file.lower().endswith(valid_extensions):
                image_paths.append(os.path.join(root, file))
                if limit and len(image_paths) >= limit:
                    return image_paths

    return image_paths

# Get paths for original and generated datasets
original_paths = get_image_paths(original_dataset, limit=1000)
generated_paths = get_image_paths(generated_dataset, limit=1000)

print(f"Number of original images: {len(original_paths)}")
print(f"Number of generated images: {len(generated_paths)}")

# Check the first few paths for both datasets
print(f"Original dataset sample paths: {original_paths[:5]}")
print(f"Generated dataset sample paths: {generated_paths[:5]}")

# Setup device and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.inception_v3(pretrained=True, transform_input=False).to(device)
model.fc = torch.nn.Identity()  # Remove the classification layer
model.eval()

# Extract Features for both datasets
print("Extracting features for the original dataset...")
original_features = extract_features(original_paths, model, device)

print("Extracting features for the generated dataset...")
generated_features = extract_features(generated_paths, model, device)

# Check if features are empty
if original_features.size == 0 or generated_features.size == 0:
    print("Error: One of the feature sets is empty. Exiting FID computation.")
else:
    # Compute FID Score
    fid_value = calculate_fid(original_features, generated_features)
    if fid_value is not None:
        print(f"FID Score: {fid_value}")


Number of original images: 1000
Number of generated images: 1000
Original dataset sample paths: ['/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train/patient52981/study1/view1_frontal.jpg', '/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train/patient13162/study44/view1_frontal.jpg', '/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train/patient13162/study44/view2_lateral.jpg', '/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train/patient13162/study15/view1_frontal.jpg', '/mnt/Internal/MedImage/unzip_chexpert_images/CheXpert-v1.0/train/patient13162/study32/view1_frontal.jpg']
Generated dataset sample paths: ['/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_Model_Imagea_generation_with_prompt/CheXpert-v1.0/train/patient01225/study5/view1_frontal.jpg', '/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_Model_Imagea_generation_with_prompt/CheXpert-v1.0/train/patient00686/study6/view2_frontal.jpg', '/mnt/Internal/MedImag

/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms, models
from tqdm import tqdm
from PIL import Image
import numpy as np
from scipy.stats import entropy

# Define the preprocessing pipeline for the Inception model
preprocess = transforms.Compose([
    transforms.Lambda(lambda x: x.convert('RGB')),  # Ensure image is in RGB format
    transforms.Resize(299),  # Resize to 299x299 (Inception-v3 input size)
    transforms.CenterCrop(299),  # Crop to 299x299
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize using ImageNet stats
])

def calculate_inception_score(features, model, device, num_classes=1000, batch_size=8):
    """
    Calculate the Inception Score (IS) of generated images using the Inception-v3 model.
    Args:
    - features (list): List of image features extracted from the model.
    - model: Pretrained Inception-v3 model.
    - device: Device (CPU or GPU) to run the model.
    - num_classes (int): Number of classes in the dataset (default is 1000 for ImageNet).
    - batch_size (int): Batch size for processing the images.
    
    Returns:
    - IS score: A float value representing the Inception Score.
    """
    model.eval()
    
    # Initialize lists to hold conditional and marginal distributions
    p_yx = []
    p_y = []
    
    # Process the image features in batches
    for i in tqdm(range(0, len(features), batch_size), desc="Calculating Inception Score"):
        batch = features[i:i + batch_size]
        
        # Modify the preprocessing loop to convert numpy arrays to PIL images
        batch = [Image.fromarray(f) if isinstance(f, np.ndarray) else f for f in batch]  # Convert ndarray to PIL Image
        batch = [preprocess(f).unsqueeze(0) for f in batch]  # Apply preprocessing to each image and add batch dimension
        batch = torch.cat(batch, dim=0).to(device)  # Concatenate the batch and move to device

        with torch.no_grad():
            # Forward pass through the Inception model
            logits = model(batch)
            p_yx_batch = F.softmax(logits, dim=1)  # p(y|x)
            
            p_yx.append(p_yx_batch.cpu().numpy())
            p_y.append(np.mean(p_yx_batch.cpu().numpy(), axis=0))  # p(y)
    
    # Concatenate all the results
    p_yx = np.concatenate(p_yx, axis=0)
    p_y = np.mean(np.concatenate(p_y, axis=0), axis=0)
    
    # Compute the Inception Score
    kl_divergence = np.mean([entropy(p_x, p_y) for p_x in p_yx])
    inception_score = np.exp(kl_divergence)
    
    return inception_score

# Setup device and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.inception_v3(pretrained=True, transform_input=False).to(device)
model.fc = torch.nn.Identity()  # Remove the classification layer

# Extract Features for both datasets (the same as FID)
print("Extracting features for the original dataset...")
original_features = extract_features(original_paths, model, device)

print("Extracting features for the generated dataset...")
generated_features = extract_features(generated_paths, model, device)

# Check if features are empty
if original_features.size == 0 or generated_features.size == 0:
    print("Error: One of the feature sets is empty. Exiting IS computation.")
else:
    # Calculate Inception Score for the generated images
    is_score = calculate_inception_score(generated_features, model, device)
    print(f"Inception Score: {is_score}")


In [ ]:
import numpy as np
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import os
from skimage.metrics import structural_similarity as ssim
from math import log10

# Paths to datasets
original_dataset = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"
generated_dataset = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/training_our_own_diffusion_model/256_diffusion_model_with_prompt/CheXpert-v1.0/train"

# Functions for calculating image quality metrics

def get_image_paths(directory, limit=None):
    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")  # Add extensions as needed
    image_paths = []

    # Traverse subdirectories to find images
    for root, _, files in os.walk(directory):  # Recursively traverse directories
        for file in files:
            if file.lower().endswith(valid_extensions):
                image_paths.append(os.path.join(root, file))
                if limit and len(image_paths) >= limit:
                    return image_paths

    return image_paths


def calculate_psnr(original_image, generated_image):
    mse = np.mean((original_image - generated_image) ** 2)
    if mse == 0:
        return 100  # Infinite PSNR for identical images
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / np.sqrt(mse))
    return psnr


def calculate_ssim(original_image, generated_image):
    return ssim(original_image, generated_image, data_range=generated_image.max() - generated_image.min(), multichannel=True)


def calculate_mse(original_image, generated_image):
    return np.mean((original_image - generated_image) ** 2)


# Inception V3 Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.inception_v3(pretrained=True, transform_input=False).to(device)
model.fc = torch.nn.Identity()  # Remove the classification layer
model.eval()

# Transform function for Inception V3 input
transform = transforms.Compose([
    transforms.Resize((299, 299)),  # Inception expects 299x299 input
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def extract_features(image_paths, model, device):
    features = []
    for image_path in tqdm(image_paths, desc="Extracting Features"):
        # Open image and convert to RGB (for color images)
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            continue
        
        # Apply transforms and extract features
        image = transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            feature = model(image).cpu().numpy().squeeze()
        features.append(feature)
    
    return np.array(features)


# Get image paths for both datasets (original and generated)
original_paths = get_image_paths(original_dataset, limit=10000)
generated_paths = get_image_paths(generated_dataset, limit=10000)

# Debugging: print number of images loaded
print(f"Number of original images: {len(original_paths)}")
print(f"Number of generated images: {len(generated_paths)}")

# If both lists are empty or too short, it suggests a path issue.
if not original_paths or not generated_paths:
    print("Check the paths or the directories for images.")

# Optionally, print first few paths to check
print("Sample original image paths:", original_paths[:5])
print("Sample generated image paths:", generated_paths[:5])

# Initialize metrics lists
psnrs = []
ssims = []
mses = []

# Loop over images and calculate metrics
for i in range(min(len(original_paths), len(generated_paths))):
    try:
        original_image = np.array(Image.open(original_paths[i]))
        generated_image = np.array(Image.open(generated_paths[i]))

        # Ensure valid pixel data
        if np.any(np.isnan(original_image)) or np.any(np.isnan(generated_image)):
            print(f"Skipping image pair {original_paths[i]} and {generated_paths[i]} due to NaN values.")
            continue
        
        # Check if images have the same dimensions
        if original_image.shape != generated_image.shape:
            print(f"Skipping image pair {original_paths[i]} and {generated_paths[i]} due to dimension mismatch.")
            continue

        # Calculate PSNR, SSIM, and MSE
        psnr = calculate_psnr(original_image, generated_image)
        ssim_value = calculate_ssim(original_image, generated_image)
        mse_value = calculate_mse(original_image, generated_image)

        psnrs.append(psnr)
        ssims.append(ssim_value)
        mses.append(mse_value)
    except Exception as e:
        print(f"Error processing image pair {original_paths[i]} and {generated_paths[i]}: {e}")

# Calculate average metrics
if psnrs and ssims and mses:
    avg_psnr = np.mean(psnrs)
    avg_ssim = np.mean(ssims)
    avg_mse = np.mean(mses)

    # Print the results
    print(f"Average PSNR: {avg_psnr:.2f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average MSE: {avg_mse:.4f}")
else:
    print("No valid image pairs to process.")